# Experimento 0 — Exploración empírica de ALE/Assault-v5

Este notebook implementa la HU001. Su objetivo es **explorar y medir** el entorno `ALE/Assault-v5` mediante una política completamente aleatoria. No entrena ningún agente.

El resultado esperado es completar la información faltante de la ficha técnica y obtener un baseline cuantitativo que sirva como referencia para seleccionar y evaluar posteriormente un algoritmo de Reinforcement Learning.

## 1. Instalación de dependencias

La celda instala únicamente las dependencias necesarias para ejecutar Gymnasium + ALE en Google Colab.

In [ ]:
!pip install -q "gymnasium[atari,accept-rom-license]" ale-py matplotlib pandas psutil

## 2. Imports y configuración central

Todos los parámetros principales del experimento se concentran aquí para facilitar su reproducción.

In [ ]:
import platform
from collections import Counter

import ale_py
import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import psutil

ENV_ID = "ALE/Assault-v5"
OBS_TYPE = "rgb"
N_EPISODES = 10
BASE_SEED = 42
FRAME_SKIP = 4
REPEAT_ACTION_PROBABILITY = 0.25

gym.register_envs(ale_py)

print(f"Python: {platform.python_version()}")
print(f"Gymnasium: {gym.__version__}")
print(f"ALE-Py: {ale_py.__version__}")
print(f"CPU: {platform.processor() or platform.machine()}")
print(f"RAM total: {psutil.virtual_memory().total / (1024**3):.2f} GB")

try:
    import torch
    gpu_available = torch.cuda.is_available()
    print(f"GPU disponible: {gpu_available}")
    if gpu_available:
        print(f"GPU: {torch.cuda.get_device_name(0)}")
        print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
except Exception as exc:
    print(f"No fue posible consultar GPU con PyTorch: {exc}")

## 3. Creación e inspección del entorno

In [ ]:
env = gym.make(
    ENV_ID,
    obs_type=OBS_TYPE,
    frameskip=FRAME_SKIP,
    repeat_action_probability=REPEAT_ACTION_PROBABILITY,
)

observation, reset_info = env.reset(seed=BASE_SEED)
env.action_space.seed(BASE_SEED)

action_meanings = env.unwrapped.get_action_meanings()

print(f"Environment ID: {ENV_ID}")
print(f"Observation type: {OBS_TYPE}")
print(f"Observation space: {env.observation_space}")
print(f"Action space: {env.action_space}")
print(f"Action meanings: {action_meanings}")
print(f"Frameskip configurado: {FRAME_SKIP}")
print(f"Repeat action probability: {REPEAT_ACTION_PROBABILITY}")
print(f"Seed base: {BASE_SEED}")
print(f"Observation shape: {observation.shape}")
print(f"Observation dtype: {observation.dtype}")
print(f"Pixel min/max: {observation.min()} / {observation.max()}")
print(f"reset() info: {reset_info}")

### Frame inicial

In [ ]:
plt.figure(figsize=(6, 8))
plt.imshow(observation)
plt.title("Frame inicial de Assault")
plt.axis("off")
plt.show()

## 4. Inspección abierta de `info`

Esta sección busca descubrir variables que ALE exponga en ejecución y que no conozcamos previamente.

In [ ]:
info_keys_seen = set(reset_info.keys())
sample_step_infos = []

for step_index in range(10):
    action = env.action_space.sample()
    observation, reward, terminated, truncated, info = env.step(action)
    info_keys_seen.update(info.keys())
    sample_step_infos.append({
        "step": step_index + 1,
        "action": int(action),
        "action_name": action_meanings[action],
        "reward": float(reward),
        "terminated": terminated,
        "truncated": truncated,
        "info": info,
    })
    if terminated or truncated:
        break

print("Claves de info descubiertas:", sorted(info_keys_seen))
for item in sample_step_infos[:5]:
    print(item)

env.close()

## 5. Función de exploración con política aleatoria

La función ejecuta episodios independientes y captura únicamente variables necesarias para el EDA.

In [ ]:
def run_random_episode(env, episode_number, seed, action_meanings):
    """Ejecuta un episodio de Assault usando una política aleatoria.

    Args:
        env: Entorno Gymnasium ya creado.
        episode_number: Número identificador del episodio.
        seed: Seed utilizada para reiniciar el entorno y espacio de acciones.
        action_meanings: Lista con los nombres de las acciones disponibles.

    Returns:
        tuple: Diccionario resumen del episodio, lista de recompensas por step,
        contador de acciones y conjunto de claves de info observadas.
    """
    observation, info = env.reset(seed=seed)
    env.action_space.seed(seed)

    info_keys = set(info.keys())
    initial_lives = info.get("lives")
    previous_lives = initial_lives
    final_lives = initial_lives
    life_changes = 0

    rewards = []
    actions = Counter()
    terminated = False
    truncated = False
    steps = 0

    while not (terminated or truncated):
        action = env.action_space.sample()
        observation, reward, terminated, truncated, info = env.step(action)

        steps += 1
        rewards.append(float(reward))
        actions[int(action)] += 1
        info_keys.update(info.keys())

        current_lives = info.get("lives")
        if current_lives is not None:
            final_lives = current_lives
            if previous_lives is not None and current_lives != previous_lives:
                life_changes += abs(int(current_lives) - int(previous_lives))
            previous_lives = current_lives

    reward_array = np.asarray(rewards, dtype=float)

    summary = {
        "episode": episode_number,
        "seed": seed,
        "total_reward": reward_array.sum(),
        "steps": steps,
        "terminated": terminated,
        "truncated": truncated,
        "initial_lives": initial_lives,
        "final_lives": final_lives,
        "life_changes": life_changes,
        "positive_reward_steps": int((reward_array > 0).sum()),
        "zero_reward_steps": int((reward_array == 0).sum()),
        "negative_reward_steps": int((reward_array < 0).sum()),
        "max_step_reward": reward_array.max() if reward_array.size else np.nan,
        "min_step_reward": reward_array.min() if reward_array.size else np.nan,
    }

    return summary, rewards, actions, info_keys


## 6. Ejecución del baseline aleatorio

Se ejecutan 10 episodios independientes usando seeds consecutivas. Puede aumentarse `N_EPISODES` desde la sección de configuración si se desea mayor precisión.

In [ ]:
env = gym.make(
    ENV_ID,
    obs_type=OBS_TYPE,
    frameskip=FRAME_SKIP,
    repeat_action_probability=REPEAT_ACTION_PROBABILITY,
)
action_meanings = env.unwrapped.get_action_meanings()

episode_results = []
all_step_rewards = []
total_action_counts = Counter()
all_info_keys = set()

for episode in range(1, N_EPISODES + 1):
    seed = BASE_SEED + episode - 1
    summary, rewards, action_counts, info_keys = run_random_episode(
        env=env,
        episode_number=episode,
        seed=seed,
        action_meanings=action_meanings,
    )
    episode_results.append(summary)
    all_step_rewards.extend(rewards)
    total_action_counts.update(action_counts)
    all_info_keys.update(info_keys)
    print(
        f"Episodio {episode:02d} | seed={seed} | "
        f"reward={summary['total_reward']:.1f} | steps={summary['steps']} | "
        f"terminated={summary['terminated']} | truncated={summary['truncated']}"
    )

env.close()

results_df = pd.DataFrame(episode_results)
results_df

## 7. Estadísticos del baseline

In [ ]:
reward_stats = results_df["total_reward"].agg(["mean", "median", "std", "min", "max"])
duration_stats = results_df["steps"].agg(["mean", "min", "max"])

total_steps = int(results_df["steps"].sum())
positive_steps = int(results_df["positive_reward_steps"].sum())
zero_steps = int(results_df["zero_reward_steps"].sum())
negative_steps = int(results_df["negative_reward_steps"].sum())

density_df = pd.DataFrame({
    "metric": [
        "positive_reward_pct",
        "zero_reward_pct",
        "negative_reward_pct",
        "avg_positive_events_per_episode",
    ],
    "value": [
        100 * positive_steps / total_steps if total_steps else np.nan,
        100 * zero_steps / total_steps if total_steps else np.nan,
        100 * negative_steps / total_steps if total_steps else np.nan,
        results_df["positive_reward_steps"].mean(),
    ],
})

print("Estadísticos de recompensa:")
display(reward_stats.to_frame("value"))
print("Estadísticos de duración:")
display(duration_stats.to_frame("value"))
print("Densidad de recompensa:")
display(density_df)

## 8. Frecuencia de acciones

In [ ]:
action_rows = []
total_actions = sum(total_action_counts.values())

for action_index, action_name in enumerate(action_meanings):
    count = total_action_counts.get(action_index, 0)
    action_rows.append({
        "action": action_index,
        "action_name": action_name,
        "count": count,
        "relative_frequency_pct": 100 * count / total_actions if total_actions else 0.0,
    })

actions_df = pd.DataFrame(action_rows)
actions_df

## 9. Vidas, terminación e información descubierta

In [ ]:
print("Claves de info observadas durante el experimento:")
print(sorted(all_info_keys))

print(f"Episodios terminated: {int(results_df['terminated'].sum())}/{len(results_df)}")
print(f"Episodios truncated: {int(results_df['truncated'].sum())}/{len(results_df)}")

if results_df["initial_lives"].notna().any():
    print("Vidas iniciales observadas:", sorted(results_df["initial_lives"].dropna().unique().tolist()))
    print(f"Pérdidas/cambios de vida promedio por episodio: {results_df['life_changes'].mean():.2f}")
    display(results_df[["episode", "initial_lives", "final_lives", "life_changes", "terminated", "truncated"]])
else:
    print("La interfaz utilizada no expuso la clave 'lives' en info durante estos episodios.")

## 10. Visualizaciones mínimas

In [ ]:
plt.figure(figsize=(8, 4))
plt.bar(results_df["episode"], results_df["total_reward"])
plt.xlabel("Episodio")
plt.ylabel("Recompensa total")
plt.title("Baseline aleatorio — recompensa por episodio")
plt.show()

plt.figure(figsize=(8, 4))
plt.bar(results_df["episode"], results_df["steps"])
plt.xlabel("Episodio")
plt.ylabel("Steps")
plt.title("Duración de episodios")
plt.show()

plt.figure(figsize=(9, 4))
plt.bar(actions_df["action_name"], actions_df["count"])
plt.xlabel("Acción")
plt.ylabel("Frecuencia")
plt.title("Frecuencia de acciones aleatorias")
plt.xticks(rotation=45)
plt.show()

if all_step_rewards:
    plt.figure(figsize=(8, 4))
    plt.hist(all_step_rewards, bins=min(30, max(5, len(set(all_step_rewards)))))
    plt.xlabel("Recompensa por step")
    plt.ylabel("Frecuencia")
    plt.title("Distribución de recompensas por step")
    plt.show()

## 11. Conclusiones del Experimento 0

La siguiente celda genera un resumen usando exclusivamente los datos obtenidos durante la ejecución.

In [ ]:
reward_mean = results_df["total_reward"].mean()
reward_std = results_df["total_reward"].std()
reward_min = results_df["total_reward"].min()
reward_max = results_df["total_reward"].max()
steps_mean = results_df["steps"].mean()
steps_min = results_df["steps"].min()
steps_max = results_df["steps"].max()
positive_pct = 100 * positive_steps / total_steps if total_steps else np.nan
zero_pct = 100 * zero_steps / total_steps if total_steps else np.nan
negative_pct = 100 * negative_steps / total_steps if total_steps else np.nan

print("CONCLUSIONES DEL EXPERIMENTO 0")
print("=" * 40)
print(f"Baseline aleatorio: {reward_mean:.2f} ± {reward_std:.2f} de recompensa sobre {len(results_df)} episodios.")
print(f"Rango de recompensa por episodio: {reward_min:.2f} a {reward_max:.2f}.")
print(f"Duración: promedio {steps_mean:.1f} steps; rango {steps_min} a {steps_max}.")
print(f"Densidad: positivos={positive_pct:.3f}%, cero={zero_pct:.3f}%, negativos={negative_pct:.3f}%.")
print(f"Terminación: {int(results_df['terminated'].sum())} terminated y {int(results_df['truncated'].sum())} truncated.")
print(f"Variables descubiertas en info: {sorted(all_info_keys)}")

if results_df["initial_lives"].notna().any():
    print(f"Vidas: valores iniciales observados {sorted(results_df['initial_lives'].dropna().unique().tolist())}; cambios promedio={results_df['life_changes'].mean():.2f}.")
else:
    print("Vidas: no se obtuvo información de vidas mediante la clave 'lives' de info.")

print("\nPreguntas de la ficha técnica respondidas:")
print("- Baseline aleatorio de recompensa y dispersión.")
print("- Rango de recompensa observado.")
print("- Duración empírica de episodios.")
print("- Densidad de recompensas positivas, cero y negativas.")
print("- Comportamiento observado de terminated/truncated.")
print("- Frecuencia real de las acciones bajo política aleatoria.")
print("- Claves adicionales expuestas por info.")

print("\nPreguntas que pueden seguir abiertas:")
if not results_df["initial_lives"].notna().any():
    print("- Número y dinámica exacta de vidas no expuesta mediante info.")
print("- Causa semántica exacta de cada evento de recompensa requiere inspección visual adicional si se necesita.")
print("- La relación entre eventos específicos del juego y recompensas no puede inferirse solo con estas métricas.")

print("\nImplicación preliminar para RL:")
if positive_pct < 1.0:
    print("Las recompensas positivas son relativamente poco frecuentes en relación con el total de steps. Esto hace relevante evaluar eficiencia muestral y fortalece el interés en métodos con Replay Buffer; PER podría resultar útil si los eventos informativos son escasos.")
else:
    print("Las recompensas positivas no resultaron extremadamente escasas en este baseline. DDQN sigue siendo atractivo por su mayor estabilidad frente a la sobreestimación de valores Q; PER debe justificarse con evidencia adicional.")
print("Esta evidencia debe combinarse con la ficha técnica antes de seleccionar definitivamente entre DDQN y DQN + PER.")